Modeling: Searches
==================

This script gives a run through of all non-linear searches that are available for modeling.

Two searches account for essentially all modeling you will do, and they are the first two documented below:

- **`Nautilus`** is the nested sampling algorithm used by every `modeling.py` example. It returns the full
  posterior -- the errors on every parameter and the covariances between them -- and is therefore the search you
  use when you need results you can quote. Extensive testing of modeling has shown it is the most accurate and
  efficient search available. For users familiar with statistical inference this may be surprising, as nested
  samplers are traditionally slower than MCMC methods such as Emcee and maximum likelihood methods such as LBFGS.
  A description of why Nautilus performs better than these other searches is beyond the scope of this script, but
  if you add me on SLACk I'd be happy to have a discussion about it!

- **`MultiStartProdigy`** is the JAX multi-start gradient optimizer used by every `start_here.py` example. It is
  far faster than Nautilus, but returns a single best-fit model with no errors at all, so it is the search you use
  to check quickly that your model and data are sensible.

Every other search documented below is an alternative you would reach for only if you really know what you are
doing, or want to cross-check a result against a different method.

Three different categories of searches are available, nested samplers (E.g. Nautilus, Dynesty), MCMC (E.g. Emcee) and
maximum likelihood (e.g. LBFGS). MCMC and MLE methods can often optionally use a "starting point" to initialize the
model-fit with the parameters where it should begin. Nested samplers do not use a starting point, but a similar
approach can be applied by putting tight priors on certain parameters.

To perform a model-fit, a fully modeling script will include steps which compose a model, create an `Analysis`
object and pass these to the search to perform the fit. We skip these steps for brevity.

__Contents__

- **Nautilus**: The recommended nested sampling algorithm, which returns the full posterior. It is gradient-free, so it does not use JAX gradients, but it does exploit JAX GPU acceleration via batched likelihood evaluation.
- **MultiStartProdigy**: The recommended JAX multi-start gradient maximum a posteriori (MAP) optimizer, which is learning-rate free and works on complex model parameter spaces.
- **Dynesty**: A nested sampling algorithm that is effective for modeling, with a lot of customization.
- **Emcee**: An ensemble MCMC sampler that is commonly used in Astronomy and Astrophysics.
- **Zeus**: An ensemble MCMC slice sampler that is the most effective MCMC method for modeling.
- **LBFGS**: A quasi-Newton optimization algorithm that is a maximum likelihood estimator (MLE) method.
- **Start Point**: An API that allows the user to specify the start-point of a model-fit, which is useful for MCMC and MLE methods.
- **Search Cookbook**: A cookbook that documents all searches available in **PyAutoFit**, including those not documented here.

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/modeling.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autogalaxy as ag

__Nautilus__

Nautilus (https://nautilus-sampler.readthedocs.io/en/latest/) is a nested sampling algorithm, and is the
recommended search for modeling. It is the search used by every `modeling.py` example in the workspace.

__Full Posterior__

Nautilus does not just return a best-fit model, it maps out the **full posterior**: the probability density of
every parameter, the errors on each one, and the covariances between them. If a fit infers an effective radius of
1.0", Nautilus tells you whether that is 1.0 +/- 0.01 or 1.0 +/- 0.5, and whether it trades off against the
`sersic_index`.

This is what most science needs, and it is why Nautilus remains the default recommendation even though the
gradient optimizer described next is faster. An optimizer hands you a single point in parameter space; Nautilus
hands you a measurement you can quote.

__JAX__

Nautilus is a **gradient-free** search. It explores parameter space by drawing new live points from a learned
boundary around the current live point set, using only evaluations of the likelihood. It therefore never
differentiates the likelihood, and unlike `MultiStartProdigy` below it neither needs nor uses JAX's gradients.

It does, however, exploit JAX on a GPU. Nautilus proposes points in batches rather than one at a time, and when
the analysis is JAX-traceable (created via `use_jax=True`) **PyAutoFit** evaluates each batch through the
`jax.vmap(jax.jit(...))` wrapped likelihood in a single call. All `n_batch` models are therefore fitted
simultaneously on the GPU, and the JAX speed-up applies in full to a Nautilus fit -- it simply comes from batched
likelihood evaluation instead of gradient descent.

`n_batch` is also the main control on GPU memory: a larger batch fits more models per call, but holds more of
them in VRAM at once.

__Live Points__

`n_live` is the main setting trading off accuracy against run-time. More live points give a more accurate result
but a longer run-time, whereas fewer are faster but risk inferring a local maxima. More complex models (that is,
models with more parameters) require more live points.

In [ ]:
search = af.Nautilus(
    path_prefix=Path("imaging", "searches"),
    name="Nautilus",
    unique_tag="example",
    # search specific settings
    n_live=100,  # The number of Nautilus "live" points, increase for more complex models.
    n_batch=50,  # GPU model fits are batched and run simultaneously, and this bounds VRAM use.
    iterations_per_quick_update=10000,
)

__MultiStartProdigy__

`MultiStartProdigy` is the recommended JAX / `optax` multi-start first-order gradient optimizer, and a maximum a
posteriori (MAP) estimator. It directly addresses the weakness of a single-start optimizer like `LBFGS` (described
below): instead of descending from a single starting point (which, for the complex parameter spaces of galaxy
models, frequently gets stuck in a local maximum), it launches `n_starts` independent optimizations from broad
starting points in parallel via `jax.vmap` and returns the best one. This wide population of starts reliably finds
the global maximum-likelihood basin, making it a robust and fast optimizer even for models where single-start
optimizers fail.

Prodigy is a *learning-rate free* update rule: it estimates its own step size as it runs, so there is no
`learning_rate` for you to tune. This is why it is the recommended default of the family:

- `MultiStartProdigy` — learning-rate free (recommended); no `learning_rate` to set.
- `MultiStartAdam` — the original of the family; robust, but you must choose a `learning_rate`.
- `MultiStartADABelief` — an Adam variant; a drop-in alternative at the same `learning_rate`.

(`MultiStartLion` is a further sign-based alternative that prefers a ~10x smaller `learning_rate`.)

Because it is gradient-based, it requires a JAX-traceable analysis (created via `use_jax=True`). It also manages
its own broad starting points, so unlike `LBFGS` it does not use the start-point API described below.

Like all optimizers it returns a single best-fit model, not a posterior with errors, so `Nautilus` above remains
the default recommendation when parameter uncertainties are required.

In [ ]:
search = af.MultiStartProdigy(
    path_prefix=Path("imaging", "searches"),
    name="MultiStartProdigy",
    n_starts=50,
    batch_size=None,  # Starts evaluated at once: `None` vmaps all 50 together, which is fastest but allocates the whole batched gradient; set an integer (e.g. 4) if you hit an out-of-memory error.
    n_steps=500,
)

__Dynesty__

Dynesty (https://github.com/joshspeagle/dynesty) is a nested sampling algorithm.

Dynesty used to be the default model-fitting algorithm, before Nautilus was found to be better. However, Dynesty with
random walk nested sampling is still an effective method for modeling and worth using if you want to check your
results with an alternative to Nautilus.

Dynesty itself supports a wide variety of different nested sampling methods, including static 
sampling (`DynestyStatic` where the number of live point is fixed), dynamic sampling (`DynestyDynamic` where the number 
of live points varies with the fit) and different approaches to point sampling (e.g. slice sampling, uniform sampling). 

If you are familiar with nested sampling you can use all dynesty's different options by customizing the code below.

In [ ]:
search = af.DynestyStatic(
    path_prefix=Path("searches"),
    name="DynestyStatic",
    unique_tag="example",
    iterations_per_quick_update=2500,
    # search specific settings
    nlive=50,
    sample="rwalk",
    walks=10,
    bound="multi",
    bootstrap=None,
    enlarge=None,
    update_interval=None,
    facc=0.5,
    slices=5,
    fmove=0.9,
    max_move=100,
)

search = af.DynestyDynamic(
    path_prefix=Path("searches"),
    name="DynestyDynamic",
    unique_tag="example",
    iterations_per_quick_update=2500,
    # search specific settings
    nlive=50,
    sample="rwalk",
    walks=10,
    bound="multi",
    bootstrap=None,
    enlarge=None,
    update_interval=None,
    facc=0.5,
    slices=5,
    fmove=0.9,
    max_move=100,
)

__Emcee__

Emcee (https://github.com/dfm/emcee) is an ensemble MCMC sampler that is commonly used in Astronomy and Astrophysics.

The wrapper with **PyAutoFit** supports different initialization methods, including a ball around the center of the
priors on the model parameters, which is the recommend initialization method for Emcee.

It also includes functionality which checks the auto correlations of the chains, and terminates the search early
if they meet certain convergence criteria. This is useful for ensuring that the chains have converged.

Whilst Emcee is a popular choice of MCMC method in astrophsyics, note that the MCMC method `Zeus`, described next, has
proven better as modeling for our tests.

In [ ]:
search = af.Emcee(
    path_prefix=Path("imaging", "searches"),
    name="Emcee",
    unique_tag="example",
    iterations_per_quick_update=5000,
    # search specific settings
    nwalkers=30,
    nsteps=500,
    initializer=af.InitializerBall(lower_limit=0.49, upper_limit=0.51),
    auto_correlations_settings=af.AutoCorrelationsSettings(
        check_for_convergence=True,
        check_size=100,
        required_length=50,
        change_threshold=0.01,
    ),
)

__Zeus__

Zeus (https://zeus-mcmc.readthedocs.io/en/latest/) is an ensemble MCMC slice sampler.

The wrapper with **PyAutoFit** supports different initialization methods, including a ball around the center of the
priors on the model parameters, which is the recommend initialization method for Emcee.

It also includes functionality which checks the auto correlations of the chains, and terminates the search early
if they meet certain convergence criteria. This is useful for ensuring that the chains have converged.

Zeus is the most effective MCMC method for modeling that we have tested, and is the recommended MCMC method,
however its performance is not as good as Nautilus.

In [ ]:
search = af.Zeus(
    path_prefix=Path("imaging", "searches"),
    name="Zeus",
    unique_tag="example",
    iterations_per_quick_update=5000,
    # search specific settings
    nwalkers=30,
    nsteps=20,
    initializer=af.InitializerBall(lower_limit=0.49, upper_limit=0.51),
    auto_correlations_settings=af.AutoCorrelationsSettings(
        check_for_convergence=True,
        check_size=100,
        required_length=50,
        change_threshold=0.01,
    ),
    tune=False,
    tolerance=0.05,
    patience=5,
    maxsteps=10000,
    mu=1.0,
    maxiter=10000,
    vectorize=False,
    check_walkers=True,
    shuffle_ensemble=True,
    light_mode=False,
)

__LBFGS__

LBFGS is a quasi-Newton optimization algorithm from scipy.

An optimizer only seeks to find the maximum likelihood model, unlike MCMC or nested sampling algorithms
like Zeus and Nautilus, which aim to map out parameter space and infer errors on the parameters. Therefore, in
principle, an optimizer like LBFGS should fit a model very fast.

In our experience, the parameter spaces fitted by models are often too complex for optimizers to be used without
careful initialization.

In [ ]:
search = af.LBFGS(
    path_prefix=Path("imaging", "searches"),
    name="LBFGS",
    unique_tag="example",
)

__Start Point__

For maximum likelihood estimator (MLE) and Markov Chain Monte Carlo (MCMC) non-linear searches, parameter space
sampling is built around having a "location" in parameter space.

This could simply be the parameters of the current maximum likelihood model in an MLE fit, or the locations of many
walkers in parameter space (e.g. MCMC).

For many model-fitting problems, we may have an expectation of where correct solutions lie in parameter space and
therefore want our non-linear search to start near that location of parameter space. Alternatively, we may want to
sample a specific region of parameter space, to determine what solutions look like there.

The start-point API allows us to do this, by manually specifying the start-point of an MLE fit or the start-point of
the walkers in an MCMC fit. Because nested sampling draws from priors, it cannot use the start-point API.

Similar behaviour can be achieved by customizing the priors of a model-fit. We could place `TruncatedGaussianPrior`'s
centred on the regions of parameter space we want to sample, or we could place tight `UniformPrior`'s on regions
of parameter space we believe the correct answer lies.

The downside of using priors is that our priors have a direct influence on the parameters we infer and the size
of the inferred parameter errors. By using priors to control the location of our model-fit, we therefore risk
inferring a non-representative model.

For users more familiar with statistical inference, adjusting ones priors in the way described above leads to
changes in the posterior, which therefore impacts the model inferred.

In [ ]:
bulge = af.Model(ag.lp_linear.Sersic)

The start-point API does not conflict with the use of priors, which are still associated with every parameter.

We manually customize the priors of the model used by the non-linear search.

We use broad `UniformPriors`'s so that our priors do not impact our inferred model and errors (which would be
the case with tight `GaussianPrior`'s.

In [ ]:
bulge.centre_0 = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
bulge.centre_1 = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
bulge.ell_comps.ell_comps_0 = af.UniformPrior(lower_limit=-0.5, upper_limit=0.5)
bulge.ell_comps.ell_comps_1 = af.UniformPrior(lower_limit=-0.5, upper_limit=0.5)
bulge.effective_radius = af.UniformPrior(lower_limit=0.5, upper_limit=1.5)
bulge.sersic_index = af.UniformPrior(lower_limit=0.5, upper_limit=6.0)

We can now compose the overall model using a `Collection`, which takes the model components we defined above.

In [ ]:
galaxy = af.Model(ag.Galaxy, redshift=0.5, bulge=bulge)

model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

We can inspect the model (with customized priors) via its `.info` attribute.

In [ ]:
print(model.info)

__Start Point__

We now define the start point of certain parameters in the model:

 - The galaxy is centred near (0.0, 0.0), so we set a start point there.

 - The size of the galaxy is around 1.0" thus we set the `effective_radius` to start here.

 - We know this galaxy is an Early-type, thus we set its `sersic_index` to start at 4.0.

For all parameters where the start-point is not specified (in this case the `ell_comps`, their 
parameter values are drawn randomly from the prior when determining the initial locations of the parameters.

In [ ]:
initializer = af.InitializerParamBounds(
    {
        model.galaxies.galaxy.bulge.centre_0: (-0.01, 0.01),
        model.galaxies.galaxy.bulge.centre_1: (-0.01, 0.01),
        model.galaxies.galaxy.bulge.effective_radius: (0.9, 1.1),
        model.galaxies.galaxy.bulge.sersic_index: (3.9, 4.1),
    }
)

The `initializer` is passed to the search (e.g. the MCMC method Emcee below), which uses it to set the start-point of 
the walkers in parameter space. 

In [ ]:
search = af.Emcee(
    path_prefix=Path("imaging", "customize"),
    name="start_point",
    nwalkers=50,
    nsteps=500,
    initializer=initializer,
)

__Search Cookbook__

There are a number of other searches supported by **PyAutoFit** and therefore which can be used, which are not
explictly documented here.

The **PyAutoFit** search cookbook documents all searches that are available, including those not documented here,
and provides the code you can easily copy and paste to use these methods.

https://pyautofit.readthedocs.io/en/latest/cookbooks/search.html